# One scientific operation per pipeline component

This is the short executable tour. Each section exposes one current public scientific operation and its declared configuration. For the comprehensive pre-MAIN review, use `research_audit.ipynb`.

The normalisation example uses the deterministic `fixture_file` transport. To perform a manual live-model audit, replace that one fixture backend object with the committed backend config shown in the output.

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np

from grammar_kt import canonical, folds, items, kc, kc_selection, kt, normalisation, qmatrix, realisation, simulation, source
from grammar_kt.canonical_schema import CANONICAL_SCHEMA, SCHEMA_PATH
from grammar_kt.io import ROOT, read_json, read_jsonl, read_yaml, write_json
from grammar_kt.records import grammar_cell

settings = read_yaml(ROOT / "experiments" / "base.yaml")
frames = {row["predicate_frame_id"]: row for row in read_jsonl(realisation.LEXICON)}

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True))

## Source

Project one raw descriptor onto exactly the five fields visible to Phase 1.

In [ ]:
source_descriptor = read_jsonl(ROOT / "modules/source/fixtures/core.jsonl")[0]
phase1_input = source.phase1_record(source_descriptor)
show({"declared_source_config": settings["source"], "raw": source_descriptor, "phase1_visible": phase1_input})

## Normalisation

Render and validate one isolated Phase-1 annotation with retained evidence. The committed live backend is displayed, but execution uses a local deterministic response.

In [ ]:
mapping_input = read_jsonl(ROOT / "modules/normalisation/fixtures/core.jsonl")[0]
method = settings["normalisation"]
tour_tmp = tempfile.TemporaryDirectory(prefix="grammar-kt-tour-")
tour_root = Path(tour_tmp.name)
fixture_response = tour_root / "normalisation-response.json"
write_json(fixture_response, {
    "egp_id": mapping_input["egp_id"], "result": "complete",
    "cells": [{"tense": "present", "aspect": "none", "voice": "active", "polarity": "positive", "clause": "declarative", "modal": "none"}],
    "note": None,
})
normalisation_result = normalisation.normalise_one(
    mapping_input,
    phase1_template=(ROOT / method["phase1_prompt"]).read_text(encoding="utf-8"),
    phase2_template=(ROOT / method["phase2_prompt"]).read_text(encoding="utf-8"),
    backend_config={"kind": "fixture_file", "response_file": str(fixture_response)},
    max_attempts=1,
    output=tour_root / "normalisation",
    phase1_only=True,
)
show({
    "declared_live_backend": read_yaml(ROOT / method["backend_config"]),
    "executed_backend": {"kind": "fixture_file", "response_file": str(fixture_response)},
    "mapping": normalisation_result["output"],
    "routing": normalisation_result["phase2_routing_reason"],
    "evidence_directory": normalisation_result["evidence_directory"],
})

## Canonical

Turn a complete mapping into exact cells, deduplicate by stable identity, and retain source edges.

In [ ]:
canonical_cells, source_edges = canonical.build([normalisation_result["output"]])
show({"canonical_schema": {"path": str(SCHEMA_PATH), "schema": CANONICAL_SCHEMA}, "cells": canonical_cells, "source_edges": source_edges})

## Realisation

Validate and realise one current WH fixture through the shared morphology/operator implementation.

In [ ]:
realisation_input = next(row for row in read_jsonl(ROOT / "modules/realisation/fixtures/core.jsonl") if row["fixture_label"] == "object_wh_lexical_do")
spec = realisation_input["spec"]
cell = grammar_cell(realisation_input["cell"])
frame = frames[spec["predicate_frame_id"]]
realisation_errors = realisation.validate_spec(spec, cell, frame, realisation_input.get("source_note"))
derivation = realisation.realise(spec, cell, frame) if not realisation_errors else None
show({"declared_lexicon": str(realisation.LEXICON), "input": realisation_input, "derivation": derivation, "errors": realisation_errors})

## Items

Use the current ontology-independent path: canonical cells → admissible item opportunities → deterministic items. No KC inventory or fold is consulted.

In [ ]:
item_config = read_json(ROOT / settings["items"]["bank_config"])
item_template = (ROOT / settings["items"]["family_prompt"]).read_text(encoding="utf-8")
item_opportunities = items.build_item_opportunities(canonical_cells, frames, item_config)
constructed_items = items.construct_items(item_opportunities, frames, item_template)
assert not any("kc" in key.lower() or "split" in key.lower() for row in constructed_items for key in row)
show({
    "declared_item_config": item_config,
    "canonical_cells": canonical_cells,
    "opportunities": item_opportunities,
    "items": constructed_items,
    "intrinsic_fingerprint": items.item_bank_fingerprint(constructed_items),
})

## KC selection

Exercise the actual development-only Phase-A selector on the bundled structural-selection fixture.

In [ ]:
selection_fixture = read_json(ROOT / "modules/kc_selection/fixtures/core.json")
selection_config = read_json(ROOT / settings["kc_selection"]["config"])
selection_result = kc_selection.evaluate_fixture(selection_fixture, selection_config)
show({
    "declared_selector_config": selection_config,
    "development_input_count": sum(row["split"] == "development" for row in selection_fixture["cell_splits"]),
    "selected_kc_ids": selection_result["selected_kc_ids"],
    "objective": selection_result["objective"],
    "held_out_content_used": selection_result["selected_policy"]["selection_metadata"]["held_out_content_used"],
})

## KC application

Apply the frozen selected policy to concrete accepted-item realizations. This is materialisation, not selection.

In [ ]:
tour_assignment = {row["canonical_cell_id"]: "development" for row in canonical_cells}
tour_runtime_items = folds.annotate_items(constructed_items, tour_assignment)
item_kc_projection, projected_kc_inventory = kc.project_items(
    tour_runtime_items, canonical_cells, selection_result["selected_policy"]
)
show({"frozen_policy": selection_result["selected_policy"], "item_projection": item_kc_projection, "projected_inventory": projected_kc_inventory})

## Q-matrix

Mechanically convert the frozen item–KC projection to columns, rows, edges, and diagnostics.

In [ ]:
q_columns, q_rows, q_edges, q_audit = qmatrix.build(
    tour_runtime_items, projected_kc_inventory, item_kc_projection
)
show({"columns": q_columns, "rows": q_rows, "edges": q_edges, "audit": q_audit})

## Simulation

Project the committed declarative structural oracle onto fixed items, then generate one learner's public events while retaining private oracle evidence separately.

In [ ]:
simulation_parameters = simulation.load_simulation_parameters(ROOT / settings["simulation"]["parameters"])
simulation_parameters["seed"] = int(settings["simulation"]["seed"])
oracle_projection, oracle_feature_ids = simulation.project_oracle_items(
    tour_runtime_items, canonical_cells, simulation_parameters
)
oracle_by_item = {row["item_id"]: row["oracle_feature_ids"] for row in oracle_projection}
event_count = len(tour_runtime_items) * int(simulation_parameters["item_passes_per_learner"])
train_end, validation_end = simulation.split_boundaries(
    event_count, simulation_parameters["train_fraction"], simulation_parameters["validation_fraction"]
)
base_events, private_oracle, learners, _learner_parameters = simulation.simulate_records(
    simulation_parameters,
    {row["item_id"]: row for row in tour_runtime_items},
    oracle_by_item,
    oracle_feature_ids,
    train_end,
    validation_end,
    target_learner="L0001",
)
show({
    "declared_oracle": simulation_parameters,
    "oracle_projection": oracle_projection,
    "learner": learners[0],
    "public_base_events": base_events,
    "private_oracle_rows_retained_separately": len(private_oracle),
})

## KT

Use the committed frozen compositional-probe fixture: development acquisition is projected once, then every probe reads the same frozen candidate state.

In [ ]:
kt_fixture = read_json(ROOT / "modules/kt/fixtures/compositional_probe.json")
acquisition, probes, supported, frozen_counts = kt.project_compositional_interactions(
    kt_fixture["acquisition_events"], kt_fixture["probe_events"], kt_fixture["item_projections"]
)
states = kt.frozen_development_statistics(acquisition)
kt_parameters = read_json(ROOT / settings["kt"]["parameters"])
_features, targets, empirical, fallback = kt.frozen_probe_features(
    probes, ["KC_COMPONENT"], states,
    alpha=float(kt_parameters["empirical"]["alpha"]),
    beta=float(kt_parameters["empirical"]["beta"]),
    cold_prior=float(kt_parameters["compositional"]["cold_kc_prior"]),
)
show({
    "declared_kt_parameters": kt_parameters,
    "acquisition_projection": acquisition,
    "probe_projection": probes,
    "development_supported_kcs": sorted(supported),
    "frozen_counts": {learner: dict(counts) for learner, counts in frozen_counts.items()},
    "empirical_probe_prediction": float(empirical[0]),
    "zero_kc_fallback": float(fallback[0]),
    "probe_updates_candidate_state": False,
})